# 03. Mô hình Image-Only & Vision-Centric cho ViMMSD

Notebook này thực hiện và đối sánh 3 bài toán trong nhóm **Image-Only / Vision-Centric**:
1. **Task 5 — Dùng CLIP Vision Encoder**: Fine-tune encoder đa phương thức tương phản (`openai/clip-vit-base-patch32`).
2. **Task 6 — Dùng ViT Image Encoder thuần**: Fine-tune Vision Transformer (`google/vit-base-patch16-224`) học trên ImageNet.
3. **Task 7 — Dùng VLM với Metadata**: Đánh giá năng lực của Vision-Language Model (Vintern-1B / Qwen2.5-VL) kết hợp OCR & mô tả ngữ cảnh hình ảnh.

In [ ]:
# === Cell bootstrap (giống nhau ở mọi notebook, xem CODEBASE.md mục 6) ===
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/yunaLee21/Multimodal_Sarcasm_Detection.git"
REF = "main"               # branch của mình, hoặc 1 commit hash để tái lặp kết quả
CODE_DIR = "/tmp/IE403"    # clone ra ngoài /kaggle/working để token không bị lưu vào output


def get_github_token():
    """Token cho repo private; repo public thì không cần tạo secret."""
    try:
        if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret("GITHUB_TOKEN")
        from google.colab import userdata
        return userdata.get("GITHUB_TOKEN")
    except Exception:
        return None


ON_CLOUD = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    if "COLAB_RELEASE_TAG" in os.environ and not os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):  # ảnh Kaggle dựng từ ảnh Colab nên cũng có biến này
        from google.colab import drive
        drive.mount("/content/drive")
    token = get_github_token()
    url = f"https://{token}@{REPO}" if token else f"https://{REPO}"
    subprocess.run(["rm", "-rf", CODE_DIR])
    subprocess.run(["git", "clone", "-q", url, CODE_DIR], check=True)
    subprocess.run(["git", "-C", CODE_DIR, "checkout", "-q", REF], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE_DIR}/requirements-kaggle.txt"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", CODE_DIR], check=True)
    if CODE_DIR + "/src" not in sys.path:
        sys.path.insert(0, CODE_DIR + "/src")  # kernel đang chạy chưa thấy package vừa cài -e
    os.chdir(CODE_DIR)
elif Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)  # local: chạy từ gốc repo (đã `pip install -e .` một lần)

print("cwd:", os.getcwd())
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()
print("code commit:", commit or "unknown")

In [ ]:
import json
import logging
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML, Image as IPImage, display

from vimmsd.utils.config import load_config
from vimmsd.training.metrics import format_metrics

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s", force=True)
logging.getLogger("httpx").setLevel(logging.WARNING)

---
## 1. Task 5: Baseline Image-only dùng CLIP (openai/clip-vit-base-patch32)

* **Kiến trúc**: CLIP Vision Tower (`clip-vit-base-patch32`) + Classification Head (MLP 2 lớp).
* **Đặc điểm**: CLIP được pretrain tương phản (contrastive) giữa cặp Image-Text trên tập dữ liệu web khổng lồ, nên vector embedding mang ngữ nghĩa trừu tượng cao, nắm bắt tốt các biểu tượng và phong cách văn hóa.

In [ ]:
# Train Image-only CLIP
# data.text.cache_file=null: image-only không dùng cache caption; nếu không có override này,
# stage_preprocessed_cache sẽ đòi attach dataset funalee/vimmsd-data-preprocessed dù không cần
!python scripts/train.py --config configs/image_only.yaml \
    --override data.text.cache_file=null \
    --note "Task 5: CLIP Image-only"

In [ ]:
cfg_clip = load_config("configs/image_only.yaml")
out_clip = Path(cfg_clip.paths.output_dir) / cfg_clip.name
if (out_clip / "test_metrics.json").exists():
    metrics_clip = json.loads((out_clip / "test_metrics.json").read_text())
    print("=== KẾT QUẢ TASK 5 (CLIP IMAGE-ONLY) ===")
    print(format_metrics(metrics_clip))
    if (out_clip / "confusion_matrix_test.png").exists():
        display(IPImage(str(out_clip / "confusion_matrix_test.png"), width=450))
else:
    print(f"Chưa có checkpoint tại {out_clip}. Hãy chạy cell train trước.")

In [ ]:
# Chi tiet qua trinh train (best epoch, so epoch da chay, thoi gian/epoch, VRAM peak)
# train.py in gia tri nay ra log nhung khong xuat file rieng -> doc lai tu checkpoint:
#   best.pt = epoch duoc chon lam model tot nhat;  last.pt = lich su moi epoch + epoch cuoi
#   results_row.md = dong train.py ghi ra: | ngay | config | commit | env | val | test | phut | note |
import re
import torch


def train_report(out_dir, label):
    out_dir = Path(out_dir)
    hist = []
    last_pt = out_dir / "last.pt"
    if last_pt.exists():
        hist = torch.load(last_pt, map_location="cpu", weights_only=False).get("history") or []
    best_pt = out_dir / "best.pt"
    best_epoch = (int(torch.load(best_pt, map_location="cpu", weights_only=False)["epoch"]) + 1) \
        if best_pt.exists() else None
    minutes = vram = None
    row_md = out_dir / "results_row.md"
    if row_md.exists():
        # file gom header + moi lan train them 1 dong -> lay dong duoi cung, khong doc header
        lines = [l for l in row_md.read_text(encoding="utf-8").splitlines() if l.strip().startswith("|")]
        if len(lines) > 1:
            row = [c.strip() for c in lines[-1].strip().strip("|").split("|")]
            try:
                minutes = float(row[6])
            except (IndexError, ValueError):
                pass
            m = re.search(r"peak VRAM ([\d.]+) GB", " ".join(row))
            vram = float(m.group(1)) if m else None
        else:
            print(f"  luu y: {row_md} chua co dong so lieu (chi co header)")
    print(f"[{label}] best epoch = {best_epoch}/{len(hist)} | epoch da chay = {len(hist)}"
          f" | tong thoi gian = {minutes} phut"
          f" | {round(minutes / len(hist), 2) if minutes and len(hist) else '?'} phut/epoch"
          f" | VRAM peak = {vram} GB")
    for h in hist:
        print(f"    epoch {h['epoch'] + 1}: train_loss={h.get('train_loss'):.4f} val_macro_f1={h['val_macro_f1']:.4f}")
    return {"best_epoch": best_epoch, "epochs_run": len(hist), "train_minutes": minutes,
            "minutes_per_epoch": round(minutes / len(hist), 2) if minutes and len(hist) else None,
            "peak_vram_gb": vram, "history": hist}


report_clip = train_report(out_clip, cfg_clip.name)
if report_clip:
    (out_clip / "train_report.json").write_text(json.dumps(report_clip, indent=2, ensure_ascii=False))

---
## 2. Task 6: Baseline Image-only dùng ViT Image Encoder thuần (google/vit-base-patch16-224)

* **Kiến trúc**: ViT thuần (`vit-base-patch16-224`, pretrain trên ImageNet-21k / 1k) + Classification Head.
* **Mục tiêu so sánh**: ViT thuần chỉ học đặc trưng thị giác (visual features, textures, objects), hoàn toàn không có alignment với ngôn ngữ tự nhiên. Thí nghiệm này là ablation study quan trọng để chứng minh giá trị của multimodal contrastive pretraining (CLIP) so với supervised vision pretraining (ViT) trong bài toán mỉa mai.

In [ ]:
# Train Image-only ViT
!python scripts/train.py --config configs/image_only_vit.yaml \
    --override data.text.cache_file=null \
    --note "Task 6: ViT Image-only"

In [ ]:
cfg_vit = load_config("configs/image_only_vit.yaml")
out_vit = Path(cfg_vit.paths.output_dir) / cfg_vit.name
if (out_vit / "test_metrics.json").exists():
    metrics_vit = json.loads((out_vit / "test_metrics.json").read_text())
    print("=== KẾT QUẢ TASK 6 (ViT IMAGE-ONLY) ===")
    print(format_metrics(metrics_vit))
    if (out_vit / "confusion_matrix_test.png").exists():
        display(IPImage(str(out_vit / "confusion_matrix_test.png"), width=450))
else:
    print(f"Chưa có checkpoint tại {out_vit}. Hãy chạy cell train trước.")

In [ ]:
# Chi tiet qua trinh train (best epoch, so epoch da chay, thoi gian/epoch, VRAM peak)
# train.py in gia tri nay ra log nhung khong xuat file rieng -> doc lai tu checkpoint:
#   best.pt = epoch duoc chon lam model tot nhat;  last.pt = lich su moi epoch + epoch cuoi
#   results_row.md = dong train.py ghi ra: | ngay | config | commit | env | val | test | phut | note |
import re
import torch


def train_report(out_dir, label):
    out_dir = Path(out_dir)
    hist = []
    last_pt = out_dir / "last.pt"
    if last_pt.exists():
        hist = torch.load(last_pt, map_location="cpu", weights_only=False).get("history") or []
    best_pt = out_dir / "best.pt"
    best_epoch = (int(torch.load(best_pt, map_location="cpu", weights_only=False)["epoch"]) + 1) \
        if best_pt.exists() else None
    minutes = vram = None
    row_md = out_dir / "results_row.md"
    if row_md.exists():
        # file gom header + moi lan train them 1 dong -> lay dong duoi cung, khong doc header
        lines = [l for l in row_md.read_text(encoding="utf-8").splitlines() if l.strip().startswith("|")]
        if len(lines) > 1:
            row = [c.strip() for c in lines[-1].strip().strip("|").split("|")]
            try:
                minutes = float(row[6])
            except (IndexError, ValueError):
                pass
            m = re.search(r"peak VRAM ([\d.]+) GB", " ".join(row))
            vram = float(m.group(1)) if m else None
        else:
            print(f"  luu y: {row_md} chua co dong so lieu (chi co header)")
    print(f"[{label}] best epoch = {best_epoch}/{len(hist)} | epoch da chay = {len(hist)}"
          f" | tong thoi gian = {minutes} phut"
          f" | {round(minutes / len(hist), 2) if minutes and len(hist) else '?'} phut/epoch"
          f" | VRAM peak = {vram} GB")
    for h in hist:
        print(f"    epoch {h['epoch'] + 1}: train_loss={h.get('train_loss'):.4f} val_macro_f1={h['val_macro_f1']:.4f}")
    return {"best_epoch": best_epoch, "epochs_run": len(hist), "train_minutes": minutes,
            "minutes_per_epoch": round(minutes / len(hist), 2) if minutes and len(hist) else None,
            "peak_vram_gb": vram, "history": hist}


report_vit = train_report(out_vit, cfg_vit.name)
if report_vit:
    (out_vit / "train_report.json").write_text(json.dumps(report_vit, indent=2, ensure_ascii=False))

---
## 3. Task 7: Dùng VLM với Metadata (Visual Language Model)

* **Bản chất**: Ảnh mỉa mai trên mạng xã hội Việt Nam thường chứa **chữ trong ảnh (text in meme)** và **ngữ cảnh hành vi phức tạp** mà các visual encoder thông thường không nắm bắt được.
* **Cách tiếp cận**: Khai thác VLM (Vintern-1B hoặc Qwen2.5-VL) kết hợp metadata (chữ OCR trích xuất + mô tả ngữ cảnh chi tiết sinh bởi VLM) để nhận định sắc thái hình ảnh.

> **Điều kiện chạy Task 7 (VLM + Metadata):**
> - Kaggle attach **2 dataset**: `phuclkz/vimmsd-private-dataset` (data + nhãn ref) và `funalee/vimmsd-data-preprocessed` (chứa `image_preprocessed.json` — OCR + mô tả VLM từ notebook `01b`, và `text_preprocessed.json`).
> - GPU ≥ 8 GB (Vintern-1B). `evaluate_llm` ghi JSONL từng mẫu nên có thể chạy tiếp khi bị ngắt.
> - Với `data.split=official` hiện tại, `test` = **private test có nhãn (~1.504 mẫu)** — metrics in ra chính là điểm trên bảng xếp hạng cuộc thi.

In [ ]:
import time
# Task 7: Vintern zero-shot ± metadata — đánh giá trên PRIVATE TEST có nhãn (split=official, ~1.504 mẫu)
# Kaggle: attach 2 dataset — phuclkz/vimmsd-private-dataset + funalee/vimmsd-data-preprocessed
#         stage_preprocessed_cache copy cache từ /kaggle/input vào /kaggle/working/cache
# Local:  stage tự skip (không có /kaggle/input); cần 2 file cache nằm sẵn trong data/processed
from vimmsd.data.dataset import load_all_records, stage_preprocessed_cache
from vimmsd.analysis.llm_compare import VinternClassifier, evaluate_llm

cfg_vlm = load_config("configs/base.yaml", [
    "data.image_text.use_ocr=true",
    "data.image_text.use_description=true",
])
stage_preprocessed_cache(cfg_vlm)

splits = load_all_records(cfg_vlm)
test_records = splits["test"]
labels = list(cfg_vlm.data.labels)
print(f"Test: {len(test_records)} mẫu | có nhãn: {sum(r['label'] >= 0 for r in test_records)}")

out_vlm = Path(cfg_vlm.paths.output_dir) / "vlm_image_centric"
out_vlm.mkdir(parents=True, exist_ok=True)

vlm = VinternClassifier("5CD-AI/Vintern-1B-v3_5", labels=labels)

# Chế độ 1: zero-shot KHÔNG có metadata (baseline VLM thuần ảnh)
_t = time.time()
metrics_no_meta, _ = evaluate_llm(
    vlm, test_records, labels,
    use_metadata=False,
    out_path=out_vlm / "vintern_no_metadata.jsonl",
)
_vlm_time = {"no_metadata": time.time() - _t}
print("=== Vintern zero-shot (không metadata) ===")
print(format_metrics(metrics_no_meta))

# Chế độ 2: zero-shot CÓ metadata (OCR + mô tả ảnh)
_t = time.time()
metrics_meta, _ = evaluate_llm(
    vlm, test_records, labels,
    use_metadata=True,
    out_path=out_vlm / "vintern_with_metadata.jsonl",
)
_vlm_time["with_metadata"] = time.time() - _t
print("=== Vintern zero-shot (có OCR + mô tả ảnh) ===")
print(format_metrics(metrics_meta))

In [ ]:
# So sinh + luu toan bo so lieu Task 7 can cho bao cao (evaluate_llm khong tu lam nhung viec nay):
#  - test_metrics_{no,with}_metadata.json  : metrics day du (co invalid_rate + confusion_matrix)
#  - confusion_matrix_test_<tag>.png       : anh CM de chen bao cao
#  - vlm_stats.json                        : thoi gian tung che do, phan bo du doan, mau tra loi khong parse duoc
# Neu metrics_no_meta/metrics_meta khong con (kernel restart) thi tu recalc tu JSONL -> khong can chay lai GPU
import time
from collections import Counter
from vimmsd.training.metrics import compute_metrics, plot_confusion_matrix

VLM_JSONL = {"no_metadata": out_vlm / "vintern_no_metadata.jsonl",
             "with_metadata": out_vlm / "vintern_with_metadata.jsonl"}
VLM_TITLES = {"no_metadata": "Vintern zero-shot (khong metadata)",
              "with_metadata": "Vintern zero-shot (+ OCR / mo ta anh)"}


def read_jsonl(path):
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def metrics_from_rows(rows):
    y_true = [labels.index(r["label"]) for r in rows]
    # cau tra loi khong parse duoc -> gan nhan sai chac chan de khong duoc tinh diem (giong evaluate_llm)
    y_pred = [labels.index(r["pred"]) if r.get("pred") else (t + 1) % len(labels)
              for r, t in zip(rows, y_true)]
    m = compute_metrics(y_true, y_pred, labels)
    m["invalid_rate"] = sum(1 for r in rows if not r.get("pred")) / max(len(rows), 1)
    return m


vlm_stats = {}
_vlm_time = globals().get("_vlm_time", {})
for tag, jsonl in VLM_JSONL.items():
    rows = read_jsonl(jsonl)
    if not rows:
        print(f"[{tag}] chua chay -> bo qua")
        continue
    mem = {"no_metadata": globals().get("metrics_no_meta"), "with_metadata": globals().get("metrics_meta")}[tag]
    m = mem if mem is not None else metrics_from_rows(rows)
    if "invalid_rate" not in m:
        m["invalid_rate"] = sum(1 for r in rows if not r.get("pred")) / len(rows)
    (out_vlm / f"test_metrics_{tag}.json").write_text(json.dumps(m, indent=2, ensure_ascii=False))
    png = out_vlm / f"confusion_matrix_test_{tag}.png"
    plot_confusion_matrix(m["confusion_matrix"], labels, png, title=VLM_TITLES[tag])

    bad = [r for r in rows if not r.get("pred")]
    dist = Counter(r.get("pred") or "INVALID" for r in rows).most_common()
    vlm_stats[tag] = {
        "n_samples": len(rows),
        "invalid_rate": m["invalid_rate"],
        "n_invalid": len(bad),
        "pred_distribution": dict(dist),
        "macro_f1": m["macro_f1"],
        "accuracy": m["accuracy"],
        "seconds": _vlm_time.get(tag),
        "minutes": round(_vlm_time[tag] / 60, 1) if _vlm_time.get(tag) else None,
        "invalid_examples": [str(r.get("raw", ""))[:200] for r in bad[:5]],
    }
    print(f"=== {VLM_TITLES[tag]} ===")
    print(f"  macro-F1 {m['macro_f1']:.4f} | acc {m['accuracy']:.4f}"
          f" | invalid {len(bad)}/{len(rows)} = {m['invalid_rate']:.1%}"
          f" | thoi gian {vlm_stats[tag]['minutes'] if vlm_stats[tag]['minutes'] else 'khong do'} phut")
    print("  phan bo du doan:", dist)
    if bad:
        print("  mau tra loi khong parse duoc:")
        for r in bad[:3]:
            print("    raw:", repr(r.get("raw"))[:160])
    print("  da luu:", out_vlm / f"test_metrics_{tag}.json", "va", png)

(out_vlm / "vlm_stats.json").write_text(json.dumps(vlm_stats, indent=2, ensure_ascii=False))
print("\nTong hop Task 7 da luu tai", out_vlm / "vlm_stats.json")

---
## 4. Tổng hợp & Đối sánh (Benchmark Table: CLIP vs ViT vs VLM)

Bảng so sánh hiệu năng của cả 3 phương pháp Image-Only / Vision-Centric.

In [ ]:
# Bang benchmark + bang per-class chi tiet (Precision/Recall/F1/Support) cho bao cao
rows_ = [("CLIP (Task 5)", out_clip / "test_metrics.json"),
         ("ViT (Task 6)", out_vit / "test_metrics.json"),
         ("VLM khong metadata (Task 7)", out_vlm / "test_metrics_no_metadata.json"),
         ("VLM + OCR/mo ta (Task 7)", out_vlm / "test_metrics_with_metadata.json")]

loaded = []
for name, metric_file in rows_:
    if Path(metric_file).exists():
        loaded.append((name, json.loads(Path(metric_file).read_text())))
    else:
        print(f"thieu ket qua: {name} ({metric_file})")

if loaded:
    results = [{
        "Phương pháp": name,
        "Macro-F1": m.get("macro_f1", 0),
        "Accuracy": m.get("accuracy", 0),
        **{f"F1 ({k})": v.get("f1", 0) for k, v in m.get("per_class", {}).items()}
    } for name, m in loaded]
    display(pd.DataFrame(results).set_index("Phương pháp").round(4))

    for name, m in loaded:
        pc = m.get("per_class", {})
        tbl = pd.DataFrame({
            "Precision": {k: v.get("precision", 0) for k, v in pc.items()},
            "Recall": {k: v.get("recall", 0) for k, v in pc.items()},
            "F1-score": {k: v.get("f1", 0) for k, v in pc.items()},
            "Support": {k: v.get("support", 0) for k, v in pc.items()},
        }).round(4)
        tbl.loc["Macro avg"] = [m.get("macro_precision", 0), m.get("macro_recall", 0),
                                m.get("macro_f1", 0), int(tbl["Support"].sum())]
        print(f"\n--- {name} ---")
        display(tbl)
        if m.get("invalid_rate") is not None:
            print(f"invalid_rate = {m['invalid_rate']:.4f}")
else:
    print("Chưa có kết quả đã train để lập bảng so sánh.")

---
## 5. Phân tích lỗi (Error Analysis)

So sánh các ca dự đoán sai để làm rõ: hình ảnh đơn lẻ thiếu thông tin gì so với khi có cả caption tiếng Việt?

In [ ]:
!python scripts/evaluate.py --config configs/image_only.yaml \
    --override data.text.cache_file=null --split test
if (out_clip / "test_predictions.csv").exists():
    pred_clip = pd.read_csv(out_clip / "test_predictions.csv")
    wrong_clip = pred_clip[pred_clip.label != pred_clip.pred]
    print(f"CLIP Image-only sai: {len(wrong_clip)}/{len(pred_clip)} mẫu ({len(wrong_clip)/len(pred_clip)*100:.1f}%)")
    display(wrong_clip.groupby(["label", "pred"]).size().unstack(fill_value=0))
    print("10 mẫu sai ngẫu nhiên:")
    display(wrong_clip.sample(min(10, len(wrong_clip)), random_state=42)[["id", "label", "pred", "caption"]])

---
## 6. Xuất số liệu cho báo cáo

Cell dưới in **toàn bộ** số liệu cần điền vào báo cáo (thời gian train, VRAM peak, best epoch,
invalid rate, per-class P/R/F1) và đóng gói output để tải về một lần.

In [ ]:
# In toan bo so lieu can cho bao cao (Task 5/6/7) duoi dang de copy
def _load(p):
    p = Path(p)
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else None


summary = {"clip": _load(out_clip / "train_report.json"),
           "vit": _load(out_vit / "train_report.json"),
           "vlm": _load(out_vlm / "vlm_stats.json")}
metrics_all = {"CLIP (Task 5)": _load(out_clip / "test_metrics.json"),
               "ViT (Task 6)": _load(out_vit / "test_metrics.json"),
               "VLM khong metadata (Task 7)": _load(out_vlm / "test_metrics_no_metadata.json"),
               "VLM + OCR/mo ta (Task 7)": _load(out_vlm / "test_metrics_with_metadata.json")}

print("=" * 78)
print("A. THONG SO KY THUAT (dien vao muc 2.2 / 3.2 / 4.2)")
print("=" * 78)
for key, label in [("clip", "Task 5 - CLIP"), ("vit", "Task 6 - ViT")]:
    r = summary.get(key) or {}
    print(f"[{label}] best epoch {r.get('best_epoch', '?')}/{r.get('epochs_run', '?')}"
          f" | tong {r.get('train_minutes', '?')} phut"
          f" | {r.get('minutes_per_epoch', '?')} phut/epoch"
          f" | VRAM peak {r.get('peak_vram_gb', '?')} GB")
for tag, s in (summary.get("vlm") or {}).items():
    print(f"[Task 7 - {tag}] {s.get('minutes', '?')} phut inference {s.get('n_samples')} mau"
          f" | invalid {s.get('n_invalid')} ({(s.get('invalid_rate') or 0):.1%})")

print()
print("=" * 78)
print("B. KET QUA PER-CLASS (dien vao muc 2.3 / 3.3 / 4.3 / 5.2)")
print("=" * 78)
for name, m in metrics_all.items():
    if not m:
        print(f"[{name}] CHUA CO KET QUA")
        continue
    print(f"\n[{name}] macro-F1={m['macro_f1']:.4f} acc={m['accuracy']:.4f}"
          + (f" invalid_rate={m['invalid_rate']:.4f}" if "invalid_rate" in m else ""))
    for k, v in m.get("per_class", {}).items():
        print(f"   {k:<15} P={v.get('precision', 0):.3f} R={v.get('recall', 0):.3f}"
              f" F1={v.get('f1', 0):.3f} n={v.get('support', 0)}")

print()
print("=" * 78)
print("C. ANH CAN CHEN (confusion matrix)")
print("=" * 78)
for p in sorted(Path(cfg_clip.paths.output_dir).rglob("confusion_matrix_test*.png")):
    print("  ", p)

In [ ]:
# Dong goi toan bo output (json + png + csv + jsonl) de tai ve 1 lan tu panel Output cua Kaggle
import shutil

out_root = Path(cfg_clip.paths.output_dir)
archive = out_root / "report_artifacts"
if archive.exists():
    shutil.rmtree(archive)
archive.mkdir(parents=True)
for p in sorted(out_root.rglob("*")):
    if p.is_file() and p.parent != archive and p.suffix in (".json", ".png", ".csv", ".jsonl", ".md"):
        shutil.copy(p, archive / f"{p.parent.name}__{p.name}")
print("Da copy", len(list(archive.iterdir())), "file vao", archive)
print("Tai ve: panel Output -> file report_artifacts.zip (hoac download ca thu muc outputs/)")
shutil.make_archive(str(out_root / "report_artifacts"), "zip", archive)
print("Zip:", out_root / "report_artifacts.zip")